> **Repository note: 08_make_demo_video.ipynb**
>
> Generates the demo videos used on slide 16 (`videos/`). Runs the trained policies in Hopper-v5 and draws each frame from MuJoCo simulation state with matplotlib, because MuJoCo's OpenGL renderer crashed in Colab. The 3× slow-motion OOD video was produced by re-running the generation cell with `FRAME_SKIP = 1` and `FPS = 40`.

# Hopper-v5 demo videos — no OpenGL required

This version does **not** use MuJoCo's graphics renderer (which crashed in Colab). It runs the real trained policies in the real Hopper-v5 simulation and draws each body part from MuJoCo's simulation state (`geom_xpos`, `geom_xmat`) with matplotlib.

Works on CPU or GPU runtimes. Runtime → Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "gymnasium[mujoco]" stable-baselines3 "imageio[ffmpeg]"

## Renderer (MuJoCo state → matplotlib frames)

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio

FRAME_SKIP = 4          # Hopper: 125 sim steps/s -> ~31 fps
FPS = 31
GEOM_COLORS = {1: "#1f4e79", 2: "#2e86ab", 3: "#2e86ab", 4: "#e07a1f"}  # torso, thigh, leg, foot


class DynamicsShiftWrapper(gym.Wrapper):
    """Same physics scaling and reset order as the training notebook."""
    def __init__(self, env, mass, friction):
        super().__init__(env)
        m = self.env.unwrapped.model
        self.base_mass, self.base_fric = m.body_mass.copy(), m.geom_friction.copy()
        self.mass, self.friction = mass, friction

    def reset(self, **kwargs):
        self.env.reset(**kwargs)
        m = self.env.unwrapped.model
        m.body_mass[:] = self.base_mass * self.mass
        m.geom_friction[:] = self.base_fric * self.friction
        try:
            import mujoco
            mujoco.mj_setConst(m, self.env.unwrapped.data)
        except Exception:
            pass
        return self.env.reset()


def capsule_segments(env):
    """Endpoints (x, z) of each Hopper capsule, read from MuJoCo's simulation state."""
    m, d = env.unwrapped.model, env.unwrapped.data
    segs = []
    for g in range(1, m.ngeom):                       # geom 0 is the floor
        c = d.geom_xpos[g]
        axis = d.geom_xmat[g].reshape(3, 3)[:, 2]     # capsule long axis
        h = m.geom_size[g, 1]
        p1, p2 = c - axis * h, c + axis * h
        segs.append((g, (p1[0], p2[0]), (p1[2], p2[2]), m.geom_size[g, 0]))
    return segs


def rollout(policy, mass, friction, seed):
    """policy(obs) -> action. Returns per-frame geometry and summary stats."""
    env = DynamicsShiftWrapper(gym.make("Hopper-v5"), mass, friction)
    obs, _ = env.reset(seed=seed)
    x0 = float(env.unwrapped.data.qpos[0])
    frames, ret, steps, done = [], 0.0, 0, False
    while not done:
        if steps % FRAME_SKIP == 0:
            frames.append((capsule_segments(env), float(env.unwrapped.data.qpos[0]) - x0, steps))
        obs, r, term, trunc, _ = env.step(policy(obs))
        ret += float(r); steps += 1; done = term or trunc
    frames.append((capsule_segments(env), float(env.unwrapped.data.qpos[0]) - x0, steps))
    env.close()
    return frames, dict(ret=ret, steps=steps, dist=frames[-1][1], fell=steps < 1000)


def make_video(clips, title, path, hold_seconds=1.5):
    """clips: list of (label, frames, stats). Writes a side-by-side MP4."""
    n = max(len(f) for _, f, _ in clips) + int(hold_seconds * FPS)
    fig, axes = plt.subplots(1, len(clips), figsize=(6.4 * len(clips), 4.2), dpi=100)
    axes = np.atleast_1d(axes)
    writer = imageio.get_writer(path, fps=FPS, macro_block_size=1)
    for i in range(n):
        for ax, (label, frames, s) in zip(axes, clips):
            segs, dist, step = frames[min(i, len(frames) - 1)]
            ended = i >= len(frames) - 1
            ax.clear()
            torso_x = segs[0][1][0]
            ax.set_xlim(torso_x - 1.6, torso_x + 1.6); ax.set_ylim(-0.1, 1.9)
            ax.set_aspect("equal"); ax.set_yticks([])
            ax.axhspan(-0.1, 0, color="#d9d9d9")
            for t in np.arange(np.floor(torso_x - 2), torso_x + 2, 0.5):   # ground marks show motion
                ax.plot([t, t], [-0.1, 0], color="#8c8c8c", lw=1)
            for g, xs, zs, r in segs:
                ax.plot(xs, zs, color=GEOM_COLORS.get(g, "k"), lw=r * 200,
                        solid_capstyle="round")
            ax.set_xlabel("x position (m)")
            status = f"step {step}  |  distance {dist:5.2f} m"
            color = "black"
            if ended:
                status = (f"FELL at step {s['steps']}  |  {s['dist']:.2f} m" if s["fell"]
                          else f"Completed 1,000 steps  |  {s['dist']:.2f} m")
                color = "#c0392b" if s["fell"] else "#1e8449"
            ax.set_title(f"{label}\n{status}", fontsize=11, color=color)
        fig.suptitle(title, fontsize=13, fontweight="bold")
        fig.tight_layout()
        fig.canvas.draw()
        writer.append_data(np.asarray(fig.canvas.buffer_rgba())[:, :, :3])
    writer.close(); plt.close(fig)


## Generate the two side-by-side videos
The printed numbers should match slide 12 / `demo_summary.csv`.

In [ ]:
import os
from stable_baselines3 import PPO, SAC

MODEL_DIR = "/content/drive/MyDrive/RL_Part2/rl_part2_final_results/models"
OUT_DIR = "/content/drive/MyDrive/RL_Part2/rl_part2_final_results/videos"
os.makedirs(OUT_DIR, exist_ok=True)

AGENTS = [  # (label, algorithm class, model file)
    ("SAC Nominal (seed 0)", SAC, "sac_nominal_seed0.zip"),
    ("PPO Randomized (seed 1)", PPO, "ppo_randomized_seed1.zip"),
]
CONDITIONS = [  # (file, title, mass, friction, eval seed from demo_summary.csv)
    ("demo_nominal.mp4", "Nominal dynamics (mass 1.0, friction 1.0)", 1.0, 1.0, 70_000),
    ("demo_severe_ood.mp4", "Severe OOD (mass 1.4 x, friction 1.5 x)", 1.4, 1.5, 71_000),
]

models = {lab: cls.load(os.path.join(MODEL_DIR, f), device="cpu") for lab, cls, f in AGENTS}

for fname, title, mass, fric, seed in CONDITIONS:
    clips = []
    for lab, _, _ in AGENTS:
        policy = lambda obs, m=models[lab]: m.predict(obs, deterministic=True)[0]
        frames, s = rollout(policy, mass, fric, seed)
        print(f"{title} | {lab}: return {s['ret']:.1f}, {s['steps']} steps, {s['dist']:.2f} m")
        clips.append((lab, frames, s))
    path = os.path.join(OUT_DIR, fname)
    make_video(clips, title, path)
    print("Saved:", path)


## Preview

In [ ]:
from IPython.display import Video, display
for f in ["demo_nominal.mp4", "demo_severe_ood.mp4"]:
    display(Video(os.path.join(OUT_DIR, f), embed=True, width=960))